<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [2]:
import folium
import pandas as pd

In [3]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [ ]:
## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [62]:
# Download and read the `spacex_launch_geo.csv`
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(spacex_csv_file)
spacex_df['Booster Version'].unique()

array(['F9 v1.0  B0003', 'F9 v1.0  B0004', 'F9 v1.0  B0005',
       'F9 v1.0  B0006', 'F9 v1.0  B0007', 'F9 v1.1', 'F9 v1.1 B1011',
       'F9 v1.1 B1010', 'F9 v1.1 B1012', 'F9 v1.1 B1013', 'F9 v1.1 B1014',
       'F9 v1.1 B1015', 'F9 v1.1 B1016', 'F9 v1.1 B1018', 'F9 FT B1019',
       'F9 FT B1020', 'F9 FT B1021.1', 'F9 FT B1022', 'F9 FT B1023.1',
       'F9 FT B1024', 'F9 FT B1025.1', 'F9 FT B1026', 'F9 v1.1  B1003',
       'F9 v1.1 B1017', 'F9 FT B1029.1', 'F9 FT B1036.1', 'F9 FT B1038.1',
       'F9 B4 B1041.1', 'F9 FT  B1036.2', 'F9 FT  B1038.2',
       'F9 B4  B1041.2', 'F9 B4  B1043.2', 'F9 FT B1031.1', 'F9 FT B1030',
       'F9 FT  B1021.2', 'F9 FT B1032.1', 'F9 FT B1034', 'F9 FT B1035.1',
       'F9 FT  B1029.2', 'F9 FT B1037', 'F9 B4 B1039.1', 'F9 B4 B1040.1',
       'F9 FT  B1031.2', 'F9 B4 B1042.1', 'F9 B5  B1046.1',
       'F9 FT  B1035.2', 'F9 B4 B1043.1', 'F9 FT  B1032.2', 'F9 B4 B1044',
       'F9 B4  B1039.2', 'F9 B4 B1045.1', 'F9 B4  B1040.2'], dtype=object)

Now, you can take a look at what are the coordinates for each site.


In [57]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
spacex_df

,Launch Site,Lat,Long,class
0,CCAFS LC-40,28.562302,-80.577356,0
1,CCAFS LC-40,28.562302,-80.577356,0
2,CCAFS LC-40,28.562302,-80.577356,0
3,CCAFS LC-40,28.562302,-80.577356,0
4,CCAFS LC-40,28.562302,-80.577356,0
5,CCAFS LC-40,28.562302,-80.577356,0
6,CCAFS LC-40,28.562302,-80.577356,0
7,CCAFS LC-40,28.562302,-80.577356,0
8,CCAFS LC-40,28.562302,-80.577356,0
9,CCAFS LC-40,28.562302,-80.577356,0


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [11]:
# Start location is NASA Johnson Space Center
from IPython.display import HTML
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)
html = site_map._repr_html_()
HTML(html)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [10]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

html = site_map._repr_html_()
HTML(html)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [16]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

locations = {"ccafs_lc": [28.562302, -80.577356],
            "ccafs_slc": [28.563197, -80.576820],
            "ksc": [28.573255, -80.646895],
            "vafb": [34.632834, -120.610745]}

for name, coord in locations.items(): 
    circle = folium.Circle(coord, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(name))
    marker = folium.map.Marker(
        coord,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html=f'<div style="font-size: 12px; color:#d35400;"><b>{name}</b></div>',
            )
        )
    site_map.add_child(circle)
    site_map.add_child(marker)

html = site_map._repr_html_()
HTML(html)

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


In [26]:
# Task 2: Mark the success/failed launches for each site on the map
counts = spacex_df.groupby('Launch Site')['class'].value_counts().unstack(fill_value=0)
counts = counts.rename(columns={1: 'Success', 0: 'Failure'})

locations = {
    "ccafs_lc": {
        "coords": [28.562302, -80.577356],
        "site": "CCAFS LC-40"
    },
    "ccafs_slc": {
        "coords": [28.563197, -80.576820],
        "site": "CCAFS SLC-40"
    },
    "ksc": {
        "coords": [28.573255, -80.646895],
        "site": "KSC LC-39A"
    },
    "vafb": {
        "coords": [34.632834, -120.610745],
        "site": "VAFB SLC-4E"
    }
}


site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for key, info in locations.items():
    
    coord = info["coords"]
    site_name = info["site"]
    
    # Lookup success/failure counts for this site
    if site_name in counts.index:
        success = int(counts.loc[site_name, 'Success'])
        failure = int(counts.loc[site_name, 'Failure'])
    else:
        success = failure = 0
    
    popup_html = f"""
    <b>{site_name}</b><br>
    Successes: {success}<br>
    Failures: {failure}
    """
    
    # Circle marker
    circle = folium.Circle(
        location=coord,
        radius=1200,
        color='#d35400',
        fill=True
    ).add_child(folium.Popup(popup_html, max_width=250))

    # Label marker
    marker = folium.Marker(
        location=coord,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html=f'<div style="font-size: 12px; color:#d35400;"><b>{key}</b></div>'
        )
    )
    
    site_map.add_child(circle)
    site_map.add_child(marker)

# Display inline (JupyterLite workaround)
HTML(site_map._repr_html_())

Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [32]:
locations = {
    "ccafs_lc": {"coords": [28.562302, -80.577356], "site": "CCAFS LC-40"},
    "ccafs_slc": {"coords": [28.563197, -80.576820], "site": "CCAFS SLC-40"},
    "ksc": {"coords": [28.573255, -80.646895], "site": "KSC LC-39A"},
    "vafb": {"coords": [34.632834, -120.610745], "site": "VAFB SLC-4E"}
}

# compute rates_table (exact strings in df)
rates_table = spacex_df.groupby('Launch Site')['class'].agg(['mean','count']).rename(columns={'mean':'success_rate','count':'n_launches'})

# helper that tries multiple match strategies
def find_rate_for_site(preferred_name):
    # 1) exact lookup
    if preferred_name in rates_table.index:
        return rates_table.loc[preferred_name, 'success_rate'], rates_table.loc[preferred_name, 'n_launches']
    # 2) lowercase exact (strip)
    idxs = [i for i in rates_table.index if i.strip().lower() == preferred_name.strip().lower()]
    if idxs:
        i = idxs[0]
        return rates_table.loc[i, 'success_rate'], rates_table.loc[i, 'n_launches']
    # 3) substring match (preferred_name words appear in df index)
    key = preferred_name.replace('_',' ').split()[0]  # try first word as fallback
    idxs = [i for i in rates_table.index if preferred_name.split()[0].lower() in i.lower() or key.lower() in i.lower()]
    if idxs:
        i = idxs[0]
        return rates_table.loc[i, 'success_rate'], rates_table.loc[i, 'n_launches']
    # 4) no match
    return None, 0

# Create map
site_map = folium.Map(location=[30, -90], zoom_start=4)

for key, info in locations.items():
    coord = info['coords']
    preferred = info.get('site', key)

    rate, nlaunch = find_rate_for_site(preferred)
    if rate is None:
        label = f"{preferred}<br>No data"
    else:
        pct = round(rate * 100, 1)
        label = f"{preferred}<br>Success Rate: {pct}%<br>({nlaunch} launches)"

    cir = folium.Circle(
        location=coord,
        radius=10000,
        color='#2a9d8f' if rate and rate >= 0.8 else '#f4a261',
        fill=True,
        fill_opacity=0.6
    ).add_child(folium.Popup(label, max_width=250))

    marker = folium.Marker(
        location=coord,
        icon=DivIcon(
            icon_size=(150, 36),
            icon_anchor=(0, 0),
            html=f'<div style="font-size:12px; color:#2a9d8f;"><b>{key}</b></div>'
        )
    )

    site_map.add_child(cir)
    site_map.add_child(marker)

# show inline (JupyterLite)
HTML(site_map._repr_html_())

Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [33]:
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [34]:

# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red

spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [45]:
# Add marker_cluster to current site_map
site_map = folium.Map(location=[30, -90], zoom_start=4)

# --- Create the marker cluster ---
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

# --- Loop through each launch record ---
for index, record in spacex_df.iterrows():
    
    # Coordinates
    lat = record['Lat']
    lon = record['Long']
    coord = [lat, lon]
    
    # Marker color based on success/failure
    color = record['marker_color']  # 'green' if class==1, 'red' if class==0
    
    # Create the marker
    marker = folium.Marker(
        location=coord,
        icon=folium.Icon(color=color, icon='rocket'),
        popup=(
            f"<b>Site:</b> {record['Launch Site']}<br>"
            f"<b>Outcome:</b> {'Success' if record['class']==1 else 'Failure'}<br>"
        )
    )
    
    # Add marker to cluster
    marker_cluster.add_child(marker)

# Display the map (for Jupyter notebooks / JupyterLite)
HTML(site_map._repr_html_())

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [47]:
# TASK 3: Calculate the distances between a launch site to its proximities
import math

def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two points
    on the Earth (specified in decimal degrees)
    Returns distance in kilometers
    """
    # convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    # haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    r = 6371  # Earth radius in km
    return c * r

locations = {
    "ccafs_lc":  [28.562302, -80.577356],
    "ccafs_slc": [28.563197, -80.576820],
    "ksc":       [28.573255, -80.646895],
    "vafb":      [34.632834, -120.610745]
}

# Create a distance matrix
dist_matrix = {}
for site1, coord1 in locations.items():
    dist_matrix[site1] = {}
    for site2, coord2 in locations.items():
        dist_matrix[site1][site2] = haversine(coord1[0], coord1[1], coord2[0], coord2[1])

df_dist = pd.DataFrame(dist_matrix)
df_dist

,ccafs_lc,ccafs_slc,ksc,vafb
ccafs_lc,0.000000,0.112447,6.899330,3825.840276
ccafs_slc,0.112447,0.000000,6.934085,3825.854446
ksc,6.899330,6.934085,0.000000,3819.052759
vafb,3825.840276,3825.854446,3819.052759,0.000000


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [48]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
HTML(site_map._repr_html_())

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [49]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [50]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
distance_coastline = calculate_distance(28.56367, -80.57163, 28.56318, -80.56799)
distance_coastline

0.35975096933889655

In [54]:

# --- Coordinates ---
coastline_point = [28.56318, -80.56799]
launch_site_point = [28.56319, -80.57681]

# --- Calculate distance ---
distance_km = haversine(coastline_point[0], coastline_point[1],
                        launch_site_point[0], launch_site_point[1])

# --- Create base map ---
site_map = folium.Map(location=coastline_point, zoom_start=15)

# --- Add marker at coastline point with distance ---
distance_marker = folium.Marker(
    location=coastline_point,
    icon=DivIcon(
        icon_size=(120, 36),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>{:10.2f} KM</b></div>'.format(distance_km)
    )
)
site_map.add_child(distance_marker)

# --- Optional: add marker for launch site ---
folium.Marker(
    location=launch_site_point,
    popup="Launch Site",
    icon=folium.Icon(color='blue', icon='rocket')
).add_to(site_map)

HTML(site_map._repr_html_())

*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [55]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
coastline_point = [28.56318, -80.56799]
launch_site_point = [28.56319, -80.57681]

# --- Calculate distance ---
distance_km = haversine(coastline_point[0], coastline_point[1],
                        launch_site_point[0], launch_site_point[1])

# --- Create base map ---
site_map = folium.Map(location=coastline_point, zoom_start=15)

# --- Add marker at coastline point with distance ---
distance_marker = folium.Marker(
    location=coastline_point,
    icon=DivIcon(
        icon_size=(120, 36),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>{:10.2f} KM</b></div>'.format(distance_km)
    )
)
site_map.add_child(distance_marker)

# --- Optional: add marker for launch site ---
folium.Marker(
    location=launch_site_point,
    popup="Launch Site",
    icon=folium.Icon(color='blue', icon='rocket')
).add_to(site_map)

folium.PolyLine(
    locations=[launch_site_point, coastline_point],
    color='red',
    weight=3,
    opacity=0.8
).add_to(site_map)

HTML(site_map._repr_html_())

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [56]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site
Titusville = [28.61269, -80.80779]
launch_site_point = [28.56319, -80.57681]

# --- Calculate distance ---
distance_km = haversine(Titusville[0], Titusville[1],
                        launch_site_point[0], launch_site_point[1])

# --- Create base map ---
site_map = folium.Map(location=Titusville, zoom_start=15)

# --- Add marker at coastline point with distance ---
distance_marker = folium.Marker(
    location=Titusville,
    icon=DivIcon(
        icon_size=(120, 36),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>{:10.2f} KM</b></div>'.format(distance_km)
    )
)
site_map.add_child(distance_marker)

# --- Optional: add marker for launch site ---
folium.Marker(
    location=launch_site_point,
    popup="Launch Site",
    icon=folium.Icon(color='blue', icon='rocket')
).add_to(site_map)

folium.PolyLine(
    locations=[launch_site_point, Titusville],
    color='red',
    weight=3,
    opacity=0.8
).add_to(site_map)

HTML(site_map._repr_html_())

Yes, the launch site keeps its distance from cities. The nearest city, Titusville, is about 23km away. This is sensible because if a rocket explodes or misses its landing area, you don't want this to happen in a populated area.

After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
